# ♻️ EcoSort AI: Deep Learning Waste Classifier & Grad-CAM XAI
### B.Tech CSE (Data Science) Capstone Major Project

This Jupyter Notebook contains the complete machine learning pipeline for the **EcoSort AI** project. It outlines:
1. **Data Preprocessing & Transformations**
2. **Transfer Learning Model Definition (MobileNetV2)**
3. **Training & Validation Loops**
4. **Explainable AI (XAI) implementation using custom Grad-CAM hooks**
5. **Visualizations of heatmaps using Matplotlib**

## 🛠️ Step 1: Import Libraries

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

## ⚙️ Step 2: Global Configurations & Device Setup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

CLASSES = [
    'Plastic', 'Paper', 'Cardboard', 'Glass', 'Metal',
    'Organic Waste', 'Clothes', 'Battery', 'Electronic Waste', 'Trash'
]
print(f"Target Classes: {CLASSES}")

## 🖼️ Step 3: Define Image Transformations
We resize all input images to `224x224` to match MobileNetV2 dimensions and apply standard ImageNet mean and standard deviation normalization.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 🧠 Step 4: Model Definition (Transfer Learning with MobileNetV2)
We load a pre-trained `MobileNetV2` feature extractor, freeze its weights, and replace the top classification head with a Dropout layer and a Linear output layer with 10 classes.

In [ ]:
class PyTorchWasteClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super(PyTorchWasteClassifier, self).__init__()
        # Load pre-trained MobileNetV2 base extractor
        weights = models.MobileNet_V2_Weights.DEFAULT
        self.base_model = models.mobilenet_v2(weights=weights)
        
        # Freeze base layer parameters
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # Replace the final linear head
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, num_classes)
        )
        
        # Unfreeze classifier parameters for fine-tuning
        for param in self.base_model.classifier.parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.base_model(x)

model = PyTorchWasteClassifier(num_classes=len(CLASSES)).to(device)
print(model)

## 🔄 Step 5: Training and Validation Loop Template

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.base_model.classifier.parameters(), lr=0.001)

def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc

## 🔍 Step 6: Explainable AI - Grad-CAM Implementation
Grad-CAM registers forward and backward hooks into the final convolutional layer of MobileNetV2 (`features[18]`) to generate activation heatmaps.

In [ ]:
class PyTorchGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.features = None
        self.gradients = None
        self.handlers = []

    def save_features(self, module, input, output):
        self.features = output

    def save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def register_hooks(self):
        self.handlers.append(self.target_layer.register_forward_hook(self.save_features))
        self.handlers.append(self.target_layer.register_full_backward_hook(self.save_gradients))

    def remove_hooks(self):
        for handle in self.handlers:
            handle.remove()
        self.handlers = []

    def generate_heatmap(self, input_tensor, target_class=None):
        self.model.eval()
        self.register_hooks()
        
        output = self.model(input_tensor)
        
        if target_class is None:
            target_class = torch.argmax(output, dim=1).item()
            
        self.model.zero_grad()
        score = output[0, target_class]
        score.backward()
        
        gradients = self.gradients.cpu().data.numpy()
        features = self.features.cpu().data.numpy()
        
        self.remove_hooks()
        
        weights = np.mean(gradients, axis=(2, 3))[0]
        
        cam = np.zeros(features.shape[2:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * features[0, i, :, :]
            
        cam = np.maximum(cam, 0)
        if np.max(cam) != 0:
            cam = cam / np.max(cam)
            
        return cam, target_class

## 🎨 Step 7: Heatmap Blending & Visualization

In [ ]:
def show_gradcam_overlay(image_path, heatmap_raw):
    # Load original image using OpenCV
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img.shape
    
    # Resize and normalize heatmap
    heatmap_resized = cv2.resize(heatmap_raw, (width, height))
    heatmap_255 = np.uint8(255 * heatmap_resized)
    
    # Color map translation
    heatmap_color = cv2.applyColorMap(heatmap_255, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    
    # Blend original image and heatmap overlay
    overlay = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)
    
    # Plot the results side-by-side
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("Explainable XAI Heatmap (Grad-CAM)")
    plt.axis("off")
    
    plt.show()

## 🧪 Step 8: Run Prediction & Grad-CAM on Test Image

In [ ]:
# Load a sample plastic bottle test image
test_image_path = 'backend/static/uploads/verify_plastic.jpg'

if os.path.exists(test_image_path):
    # Preprocess test image
    image = Image.open(test_image_path).convert('RGB')
    input_tensor = val_transform(image).unsqueeze(0).to(device)
    
    # Target final conv block in MobileNetV2
    target_layer = model.base_model.features[18]
    
    # Run Grad-CAM
    with torch.enable_grad():
        input_tensor.requires_grad_()
        grad_cam_engine = PyTorchGradCAM(model, target_layer)
        heatmap_raw, predicted_idx = grad_cam_engine.generate_heatmap(input_tensor)
        
    print(f"Predicted Class: {CLASSES[predicted_idx]}")
    show_gradcam_overlay(test_image_path, heatmap_raw)
else:
    print(f"Please upload an image to {test_image_path} to test prediction and Grad-CAM plotting!")